# Data generation

In [1]:
N = 80 # number of qubits
#N = parse(Int, ARGS[1])

num_data = 1000 # number of data
nsamp = 500 # number of shots

500

In [2]:
using SparseArrays
using LinearAlgebra
using KrylovKit
using Arpack
using Plots
using CSV
using DataFrames
using Printf
using PyCall
using ITensors, ITensorMPS
using ProgressBars
using Random
using JLD2

const pynp = pyimport("numpy")

PyObject <module 'numpy' from '/home/chinzei/.julia/conda/3/lib/python3.10/site-packages/numpy/__init__.py'>

In [29]:
BLAS.set_num_threads(1) 
ITensors.blas_get_num_threads() 

1

In [30]:
# original functions for shadow
include("my_cs_tools_v250625.jl")

TESK (generic function with 1 method)

# Preliminaries

In [36]:
function ITensors.op(::OpName"Sdag", ::SiteType"S=1/2", s1::Index)
  mat = [1 0 
         0 -im]
  return itensor(mat, s1', s1)
end
#"""#
                        
function haar_random(dim::Int)    
                            
    #Random.seed!(123456789+seed)
                            
    a=randn(dim,dim)
    b=randn(dim,dim)
    c = a+b*im
                            
    # QR decomposition
    QRFactorization = qr(c)

    # Q-matrix
    Q = QRFactorization.Q

    # R-matrix
    R = QRFactorization.R

    # Lambda-matrix
    Lambda = zeros(ComplexF64, dim, dim)
    for i=1:dim
        Lambda[i,i] = R[i,i]/abs(R[i,i])
    end

    # haar random
    U = Q*Lambda
    return U            
end

haar_random (generic function with 1 method)

# DMRG

In [43]:
function XXZ_dmrg(J, delta, N)
    sites = siteinds("S=1/2",N)

    ampo = OpSum()
    for j=1:N-1
        if j%2 == 1
            ampo += 1.0,"X",j,"X",j+1
            ampo += 1.0,"Y",j,"Y",j+1
            ampo += 1.0*delta,"Z",j,"Z",j+1
        else
            ampo += 1.0*J,"X",j,"X",j+1
            ampo += 1.0*J,"Y",j,"Y",j+1
            ampo += 1.0*J*delta,"Z",j,"Z",j+1
        end
    end
    H = MPO(ampo,sites)

    psi0 = randomMPS(sites,10)

    sweeps = Sweeps(5)
    setmaxdim!(sweeps, 10,20,100,100,200)
    setcutoff!(sweeps, 1E-10)

    energy, psi = dmrg(H,psi0, sweeps)
    
    return energy, psi, sites
end

XXZ_dmrg (generic function with 1 method)

# Inversion symmetric noise

In [44]:
function inversion_symmetric_noise(psi::MPS, sites::Vector{Index{Int64}})
    
    mat = [haar_random(2) for i=1:div(N,2)]
    gates = ITensor[]
    for i=1:N
        if i <= div(N,2)
            s1 = sites[i]
            op = itensor(mat[i], s1', s1)
            push!(gates, op)
        else
            s1 = sites[i]
            op = itensor(mat[N-i+1], s1', s1)
            push!(gates, op)
        end
    end

    psi_noisy = ITensors.apply(gates, psi; cutoff = 1e-10)
    normalize!(psi_noisy)
        
    return psi_noisy, sites
end

inversion_symmetric_noise (generic function with 1 method)

# Data generation

In [45]:
function create_data(N::Int64, seed0::Int64)
    Random.seed!(seed0)
    
    J = 1.8*rand()+0.1
    delta = 0.5
    
    label = 0
    if J<1
        label = 0
    else
        label = 1
    end
    
    energy, psi, sites = XXZ_dmrg(J, delta, N)
    psi_noisy, sites = inversion_symmetric_noise(psi, sites)
    
    return psi_noisy, label
end

create_data (generic function with 1 method)

In [46]:
# MPS
seed_list = [i for i=1:num_data]
for seed in seed_list
    psi, label = create_data(N, seed) 
    save( "data_N=$(N)/qstate_seed=$(seed).jld2", Dict("psi"=>psi, "label"=>label) )
end

After sweep 1 energy=-100.80253064224046  maxlinkdim=10 maxerr=1.31E-03 time=0.169
After sweep 2 energy=-100.8025707864811  maxlinkdim=8 maxerr=9.64E-11 time=0.127
After sweep 3 energy=-100.80257078648656  maxlinkdim=8 maxerr=2.95E-11 time=0.130
After sweep 4 energy=-100.80257078648665  maxlinkdim=8 maxerr=2.95E-11 time=0.150
After sweep 5 energy=-100.80257078648688  maxlinkdim=8 maxerr=2.95E-11 time=0.129
After sweep 1 energy=-100.15700164786692  maxlinkdim=10 maxerr=8.34E-04 time=0.172
After sweep 2 energy=-100.15710564916591  maxlinkdim=8 maxerr=9.99E-11 time=0.125
After sweep 3 energy=-100.15710564876593  maxlinkdim=8 maxerr=6.64E-11 time=0.135
After sweep 4 energy=-100.15710564876625  maxlinkdim=8 maxerr=6.63E-11 time=0.113
After sweep 5 energy=-100.15710564876579  maxlinkdim=8 maxerr=6.63E-11 time=0.118
After sweep 1 energy=-108.55290950152691  maxlinkdim=10 maxerr=1.95E-03 time=0.139
After sweep 2 energy=-108.56250430299009  maxlinkdim=20 maxerr=3.16E-10 time=0.248
After sweep 3

In [ ]:
# CS
for seed in seed_list
    psi = load("data_N=$(N)/qstate_seed=$(seed).jld2","psi")
    label = load("data_N=$(N)/qstate_seed=$(seed).jld2","label")
    cs = random_measurement(N,nsamp,psi)
    save( "data_N=$(N)/cs_nsamp=$(nsamp)_seed=$(seed).jld2", Dict("cs"=>cs, "label"=>label) )
end

In [ ]:
# label
label_list = []
for seed in seed_list
    label = load("data_N=$(N)/qstate_seed=$(seed).jld2","label")
    push!(label_list, label)
end
filename = @sprintf("gram_matrix_N=%d/label_ndata=%d.csv", N, num_data)
pynp.savetxt(filename, label_list, delimiter=",")